<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="350px" align="right" border="0"><br>

# Object Oriented Programming

**_Financial Applications_**

Dr. Yves J. Hilpisch | The Python Quants GmbH

<img src="http://hilpisch.com/images/tpq_bootcamp.png" width=350px align=left>

## Case Study: Implementing a Class for Monte Carlo Option Valuation

The class shall be based on the geometric Brownian motion as its financial model ("Black-Scholes-Merton (1973)") and shall:

* allow for appropriate parmetrization
* allow for the standardized drawing of pseudo-random numbers (with moment matching)
* allow for an efficient (and lazy) simulation of paths
* allow for the visualization of some simulated paths
* allow for the lazy evaluation of European put and call options
* allow for a "proper" updating of all attribute values (by "set_xyz" methods) -- as an exercise!

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_addon.git
import sys
sys.path.append('python_for_algo_trading_addon')


In [ ]:
import math
import numpy as np
from pylab import plt
plt.style.use('seaborn-v0_8')
%matplotlib inline

The SDE of the Black-Scholes-Merton model is:

$$
dS_t = r S_t dt + \sigma  S_t dZ_t
$$

The difference equation for simulation is:

$$
S_t = S_{t-\Delta t} \cdot e^{\left(r - \frac{\sigma^2}{2} \right) \Delta t + \sigma \sqrt{\Delta t} z}
$$

with $z$ being a standard normally distributed rv.

Monte Carlo estimator for European put option:

$$
P_0 = e^{-rT} \frac{1}{I} \sum_i^I \max[K - S_T^i, 0]
$$

with $I$ being the number of paths simulated ans $S_T^i$ being a single simulated value of the underlying at maturity.

In [ ]:
initial_value = 36.  # S0
maturity = 1.0  # T  -- in year fractions
volatility = 0.2  # sigma
short_rate = 0.06  # r

In [ ]:
steps = 50
paths = 10000

In [ ]:
interval = maturity / steps

In [ ]:
interval

In [ ]:
np.random.standard_normal((5, 5))

In [ ]:
class monte_carlo_valuation(object):
    def __init__(self, initial_value, maturity, volatility, short_rate, steps, paths):
        self.initial_value = initial_value
        self.maturity = maturity
        self.volatility = volatility
        self.short_rate = short_rate
        self.steps = steps
        self.paths = paths
        self.interval = self.maturity / self.steps
        
    def draw_random_numbers(self, moment_matching=True):
        rn = np.random.standard_normal((self.steps + 1, self.paths))
        if moment_matching is True:
            rn -= rn.mean()
            rn /= rn.std()
        return rn
    
    def simulate_paths(self):
        rn = self.draw_random_numbers()
        self.data = np.zeros_like(rn)
        self.data[0] = self.initial_value
        for t in range(1, self.steps + 1):
            self.data[t] = self.data[t - 1] * np.exp(
                (self.short_rate - 0.5 * self.volatility ** 2) * self.interval +
                 self.volatility * math.sqrt(self.interval) * rn[t])
            
    def plot_paths(self, count=10):
        try:
            self.data
        except:
            self.simulate_paths()
        plt.figure(figsize=(10, 6))
        plt.plot(mcv.data[:, :count])
        
    def value_option(self, option, strike):
        try:
            self.data
        except:
            self.simulate_paths()
        if option == 'put':
            value = math.exp(-self.short_rate * self.maturity) * np.maximum(strike - self.data[-1],
                                                                        0).mean()
        elif option == 'call':
            value = math.exp(-self.short_rate * self.maturity) * np.maximum(self.data[-1] - strike,
                                                                        0).mean()
        else:
            return 'Option type not known.'
        return value

In [ ]:
mcv = monte_carlo_valuation(initial_value, maturity, volatility, short_rate, steps, paths)

In [ ]:
mcv.short_rate

In [ ]:
mcv.draw_random_numbers().mean()

In [ ]:
mcv.draw_random_numbers().std()

In [ ]:
mcv.plot_paths(5)

In [ ]:
mcv.simulate_paths()

In [ ]:
mcv.data

In [ ]:
mcv.data.nbytes  # size of the data set in bytes

In [ ]:
mcv.plot_paths(15)

In [ ]:
mcv.value_option('put', 40.)

In [ ]:
mcv.value_option('call', 40.)

In [ ]:
strikes = np.linspace(35, 45, 100)

In [ ]:
strikes

In [ ]:
%time put_values = [mcv.value_option('put', strike) for strike in strikes]

In [ ]:
put_values[:10]

## Case Study: Implementing a Class for Mean-Variance Portfolio Theory

The class shall:

* receive data for symbols
* calculate log returns
* calculate portfolio return for given weights
* calculate portfolio volatility for given weights
* allow for the adding of symbols ...
* ... and the removal of a symbol
* allow for a Monte Carlo simulation over portfolio weights
* allow for a plotting of the resulting volatility-return combinations

In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv('http://hilpisch.com/tr_eikon_eod_data.csv',
                                index_col=0, parse_dates=True)

In [ ]:
# data from Thomson Reuters Eikon API
data.info()

In [ ]:
data['.VIX'].plot(figsize=(10, 6));

In [ ]:
nos = 3

In [ ]:
nos * [1 / nos]

In [ ]:
w = np.random.random((10, 2))
w

In [ ]:
c = np.array(2 * [w.sum(axis=1)]).T

In [ ]:
w / c

In [ ]:
class mean_variance_portfolio(object):
    def __init__(self, symbols, weights=None):
        self.symbols = symbols
        self.nos = len(symbols)
        if weights is None:
            self.weights = np.array(self.nos * [1 / self.nos])
        else:
            self.weights = weights
        self.retrieve_data()
        
    def retrieve_data(self):
        self.data = pd.read_csv('http://hilpisch.com/tr_eikon_eod_data.csv',
                                index_col=0, parse_dates=True)
        self.data = self.data[self.symbols]
        self.returns = np.log(self.data / self.data.shift(1)).dropna()
        self.means = self.returns.mean() * 252
        self.stds = self.returns.std() * 252 ** 0.5
        self.cov = self.returns.cov() * 252
        
    def calculate_portfolio_return(self, weights=None):
        if weights is None:
            weights = self.weights
        return np.dot(self.means, weights)
    
    def calculate_portfolio_volatility(self, weights=None):
        if weights is None:
            weights = self.weights
        return np.dot(weights, np.dot(self.cov, weights)) ** 0.5
    
    def simulate_portfolio_weights(self, runs):
        weights = np.random.random((runs, self.nos))
        corr = np.array(self.nos * [weights.sum(axis=1)]).T
        return weights / corr
    
    def plot_volatility_mean(self, runs):
        weights = self.simulate_portfolio_weights(runs)
        vm = [(self.calculate_portfolio_volatility(weight),
               self.calculate_portfolio_return(weight)) for weight in weights]
        vm = np.array(vm)
        plt.figure(figsize=(10, 6))
        plt.plot(vm[:, 0], vm[:, 1], 'o')
        plt.xlabel('volatility')
        plt.ylabel('mean return')
        plt.title(' | '.join(self.symbols))

In [ ]:
symbols = ['AAPL.O', 'MSFT.O', 'AMZN.O']

In [ ]:
mvp = mean_variance_portfolio(symbols)

In [ ]:
mvp.weights

In [ ]:
mvp.data.info()

In [ ]:
mvp.returns.head()

In [ ]:
mvp.means

In [ ]:
mvp.stds

In [ ]:
mvp.cov

In [ ]:
mvp.calculate_portfolio_return()

In [ ]:
np.dot(mvp.means, mvp.weights)

In [ ]:
# mvp.calculate_portfolio_return(weights=[0.7, 0.3])

In [ ]:
mvp.calculate_portfolio_volatility()

In [ ]:
# mvp.calculate_portfolio_volatility(weights=[0.7, 0.3])

In [ ]:
mvp.simulate_portfolio_weights(50).sum(axis=1)

In [ ]:
mvp.plot_volatility_mean(500)

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="350px" align="right" border="0"><br>